In [1]:
import os
os.chdir("/Users/student/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/1. LSE/Postgraduate/Growth Lab/SarahRashid0.github.io/COTW")
import altair as alt
import numpy as np
import pandas as pd
import requests
import eco_style
alt.theme.enable("light")


ThemeRegistry.enable('light')

In [3]:
# ── Data ─────────────────────────────────────────────────────────────────────
gcse_url = "https://raw.githubusercontent.com/SarahRashid0/SarahRashid0.github.io/refs/heads/main/website/gcse_results_by_institution.csv"
alevel_url = "https://raw.githubusercontent.com/SarahRashid0/SarahRashid0.github.io/refs/heads/main/website/alevel_results_by_intitutions.csv"

def load_and_melt(url, above_a_prefix, above_c_prefix):
    df = pd.read_csv(url)
    type_map = {
        "All state-funded": "State-funded",
        "Independent school including city training colleges (CTCs)": "Independent"
    }
    df = df[df["Centre type"].isin(type_map.keys())].copy()
    df["school_type"] = df["Centre type"].map(type_map)

    results = {}
    for prefix, grade in [(above_a_prefix, "A"), (above_c_prefix, "C")]:
        cols = [c for c in df.columns if c.startswith(prefix)]
        melted = df[["school_type"] + cols].melt(
            id_vars="school_type", var_name="metric_year", value_name="value"
        )
        melted["year"] = melted["metric_year"].str.split("_").str[1]
        melted["percentage"] = melted["value"].str.replace("%", "").astype(float)
        # Add gap
        pivot = melted.pivot(index="year", columns="school_type", values="percentage").reset_index()
        pivot["gap"] = pivot["Independent"] - pivot["State-funded"]
        melted = melted.merge(pivot[["year", "gap"]], on="year")
        results[grade] = (melted, pivot)
    return results

gcse = load_and_melt(gcse_url, "AboveA_", "AboveC_")
alevel = load_and_melt(alevel_url, "AboveA_", "AboveC_")

year_order = ["2019", "2020", "2021", "2022", "2023", "2024", "2025"]

# ── Chart builder ─────────────────────────────────────────────────────────────
def make_chart(long_df, pivot_df, title, show_legend=False):
    area = alt.Chart(pivot_df).mark_area(opacity=0.08, color="#122b39").encode(
        x=alt.X("year:O", sort=year_order),
        y=alt.Y("State-funded:Q", scale=alt.Scale(domain=[0, 100])),
        y2=alt.Y2("Independent:Q")
    )

    lines = alt.Chart(long_df).mark_line(
        point=alt.OverlayMarkDef(filled=True, size=35),
        strokeWidth=2.5
    ).encode(
        x=alt.X("year:O", sort=year_order),
        y=alt.Y("percentage:Q", scale=alt.Scale(domain=[0, 100]), title=None),
        color=alt.Color(
            "school_type:N",
            scale=alt.Scale(
                domain=["Independent", "State-funded"],
                range=["#179fdb", "#e6224b"]
            ),
            legend=alt.Legend(title="School type") if show_legend else None
        ),
        tooltip=[
            alt.Tooltip("year:O", title="Year"),
            alt.Tooltip("school_type:N", title="School type"),
            alt.Tooltip("percentage:Q", title="%", format=".1f"),
            alt.Tooltip("gap:Q", title="Gap (pp)", format=".1f")
        ]
    )

    return (area + lines).properties(
        width=260, height=200,
        title=alt.TitleParams(text=title, fontSize=12)
    )

# ── Four charts ───────────────────────────────────────────────────────────────
gcse_c  = make_chart(*gcse["C"],  "GCSE: Grade 5+ (C) in all subjects")
gcse_a  = make_chart(*gcse["A"],  "GCSE: Grade 7+ (A) in all subjects")
alevel_c = make_chart(*alevel["C"], "A-level: Grade C+ in all subjects")
alevel_a = make_chart(*alevel["A"], "A-level: Grade A+ in all subjects", show_legend=True)

# ── 2x2 grid ─────────────────────────────────────────────────────────────────
chart = (
    (gcse_c | gcse_a) & (alevel_c | alevel_a)
).properties(
    title=alt.TitleParams(
        text="1. Independent schools outperform state schools across all exam types",
        subtitle="England 2019–2025   |   * 2020–21 reflect teacher-assessed grades",
        fontSize=15, subtitleFontSize=12, anchor="start"
    )
).configure_concat(spacing=25)

chart

alt.VConcatChart(...)